# LeWM Duckietown Retrain (Exploratory Data)

This notebook is the current **active** path for exploratory-data experiments.

Workflow:
1. Pull `duckie_explore.h5`
2. Run the encoder-based `obs -> action` probe gate
3. Proceed to training only if steering `val R² < 0.6`


In [ ]:
# Config
DATA_NEW_S3 = 's3://leworldduckie/data/duckie_explore.h5'
DATA_OLD_S3 = 's3://leworldduckie/data/duckietown_100k.h5'
DATA_NEW_LOCAL = '/content/duckie_explore.h5'
DATA_OLD_LOCAL = '/content/duckietown_100k.h5'

# Use the fs6 checkpoint (or update if newer)
CKPT_S3 = 's3://leworldduckie/training/runs/notebook/checkpoint_best.pt'
CKPT_LOCAL = '/content/lewm_checkpoint.pt'

# Probe gate threshold
STEER_R2_GATE = 0.6


In [ ]:
# Colab setup
import subprocess

def sh(cmd):
    print('>', cmd)
    subprocess.check_call(['bash', '-lc', cmd])

sh('pip install -q boto3 h5py scikit-learn torch torchvision')
sh('git clone --depth 1 https://github.com/giuliovv/leworldduckie.git /content/leworldduckie || true')


In [ ]:
# Download datasets + checkpoint from S3
import boto3
from urllib.parse import urlparse

def s3_download(uri, local):
    u = urlparse(uri)
    boto3.client('s3', region_name='us-east-1').download_file(u.netloc, u.path.lstrip('/'), local)
    print('downloaded', uri, '->', local)

s3_download(DATA_NEW_S3, DATA_NEW_LOCAL)
s3_download(DATA_OLD_S3, DATA_OLD_LOCAL)
s3_download(CKPT_S3, CKPT_LOCAL)


In [ ]:
# Probe gate (encoder mode default)
# IMPORTANT: Do not start retraining unless steering val R² < 0.6

cmd = f'''
cd /content/leworldduckie && python src/probe_obs_to_action.py \
  --mode encoder \
  --ckpt {CKPT_LOCAL} \
  --data {DATA_NEW_LOCAL} \
  --baseline-data {DATA_OLD_LOCAL} \
  --max-samples 50000 \
  --epochs 8 \
  --batch-size 256
'''

import subprocess
subprocess.check_call(['bash', '-lc', cmd])


## Training (run only if gate passes)

If steering `val R² < 0.6`, continue with your training notebook cells or scripts.

Example script entrypoint:
```bash
cd /content/leworldduckie
python src/train.py
```
